# Topography generation timings

Sibling notebook to `time_generation_design.ipynb`, but for the
**topography** pipeline at `scripts/base_plan/2.0/`. That pipeline
produces ONE hatched DXF per input (not four), routed automatically
to either the MSK or the regions sub-pipeline based on the source
drawing's layer naming.

Why a separate notebook:

* Different pipeline (`base_plan/2.0/pipeline.py`, not `design_plan/pipeline`).
* Different output shape (one DXF, not four → simpler timing table).
* Different input folder — point `TOPO_DIR` below at YOUR topography DXFs.

What's measured per drawing:

| Column | Meaning |
|---|---|
| `source` | Auto-detected sub-pipeline (`msk` / `regions`) or whatever `FORCE_SOURCE` overrode it to |
| `total` | End-to-end time of `pipeline.run(...)` — extract + classify + hatch generation + save |

The base-plan pipeline is monolithic (no clean stage boundaries inside
`msk.run` / `regions.run`), so we record `total` only. The area-stats
cell at the bottom does the same source-vs-generated hull comparison
as in the design notebook.

Pre-flight: run this notebook in a **fresh kernel** if the design
notebook was running earlier — both pipelines define modules called
`pipeline`, and we work around the namespace collision via
`importlib.util`, but a clean kernel is the safest path.

In [ ]:
import importlib.util
import sys
import time
from pathlib import Path

import pandas as pd

HERE = Path.cwd()
BASE_PLAN_ROOT = (HERE.parent / 'base_plan' / '2.0').resolve()
if not BASE_PLAN_ROOT.is_dir():
    raise FileNotFoundError(
        f'Cannot find base_plan at {BASE_PLAN_ROOT}. Move this notebook back '
        f'into scripts/design_plan/ or adjust BASE_PLAN_ROOT manually.'
    )
if str(BASE_PLAN_ROOT) not in sys.path:
    sys.path.insert(0, str(BASE_PLAN_ROOT))

_spec = importlib.util.spec_from_file_location(
    'base_plan_pipeline',
    BASE_PLAN_ROOT / 'pipeline.py',
)
base_pipeline = importlib.util.module_from_spec(_spec)
sys.modules['base_plan_pipeline'] = base_pipeline
_spec.loader.exec_module(base_pipeline)

from pipelines import DEFAULT_MODEL_DIR

print(f'base_pipeline.run imported from {BASE_PLAN_ROOT / "pipeline.py"}')
print(f'model dir = {DEFAULT_MODEL_DIR}')

base_pipeline.run imported from C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\base_plan\2.0\pipeline.py
model dir = C:\Users\artem\MAIN\projects\plain\local\python_modules\ai_models


In [ ]:
TOPO_DIR = Path(r'C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\time_test_topo')

EXCEL_PATH       = HERE / 'timings' / 'topo_timings.xlsx'
AREAS_EXCEL_PATH = HERE / 'areas'   / 'topo_area_stats.xlsx'
EXCEL_PATH.parent.mkdir(exist_ok=True, parents=True)
AREAS_EXCEL_PATH.parent.mkdir(exist_ok=True, parents=True)

DO_WARMUP = True

# None for auto source detection;
# set to 'msk' or 'regions' to short-circuit.
FORCE_SOURCE = None

print(f'Input dir   : {TOPO_DIR}')
print(f'Excel out   : {EXCEL_PATH}')
print(f'Areas out   : {AREAS_EXCEL_PATH}')
print(f'Warmup      : {DO_WARMUP}')
print(f'Force route : {FORCE_SOURCE}')

Input dir   : C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\time_test_topo
Excel out   : c:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\timings\topo_timings.xlsx
Areas out   : c:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\areas\topo_area_stats.xlsx
Warmup      : True
Force route : None


In [ ]:
# Find input DXFs
if not TOPO_DIR.is_dir():
    raise FileNotFoundError(
        f'TOPO_DIR does not exist: {TOPO_DIR}\n'
        f'Edit the Config cell above and point it at the folder holding '
        f'your topography .dxf files.'
    )

dxf_paths = sorted(p for p in TOPO_DIR.glob('*.dxf') if p.is_file())
if not dxf_paths:
    raise FileNotFoundError(f'No .dxf files found directly in {TOPO_DIR}')

print(f'Found {len(dxf_paths)} DXF file(s):')
for p in dxf_paths:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f'  {p.name:<40s}  {size_mb:6.2f} MB')

Found 10 DXF file(s):
  104-11-2024ПВВ-ИГДИ-Г.3..dxf               47.23 MB
  106-11-2024ПВВ-ИГДИ-Г.3.dxf                 7.58 MB
  93-10-2024ПВВ-ИГДИ-Г.3.dxf                 12.74 MB
  ГЕО_Вишневый сад.dxf                        2.16 MB
  Зарайск_итог 10_02_2021_итог.dxf           33.96 MB
  ИГДИ-Г.33213.dxf                            6.52 MB
  Лотошино готово.dxf                        32.31 MB
  Оленегорск ТГР.dxf                          0.43 MB
  Пушкино 500.dxf                            21.63 MB
  Топография.dxf                             17.53 MB


In [ ]:
def time_one_topo(input_path, force_source=FORCE_SOURCE,
                   model_dir=DEFAULT_MODEL_DIR, verbose=True):
    input_path = Path(input_path).resolve()
    output_dir = input_path.parent / input_path.stem
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f'{input_path.stem}_topo.dxf'

    if verbose:
        print(f'\n=== {input_path.name} ===')

    t0 = time.perf_counter()
    result = base_pipeline.run(
        str(input_path),
        force_source=force_source,
        output_path=str(output_path),
        model_dir=model_dir,
        no_xlsx=True, 
        draw_tep=False, 
    )
    elapsed = time.perf_counter() - t0

    times = {
        'drawing': input_path.name,
        'source':  result.get('source', 'unknown'),
        'total':   elapsed,
    }
    if verbose:
        print(f'  source : {times["source"]}')
        print(f'  total  : {times["total"]:7.2f}s')
    return times

print('time_one_topo() defined.')

time_one_topo() defined.


In [ ]:
raw_rows = []
for i, p in enumerate(dxf_paths, start=1):
    print(f'\n[{i}/{len(dxf_paths)}] timing {p.name}')
    try:
        raw_rows.append(time_one_topo(p, verbose=True))
    except Exception as e:
        print(f'  FAILED on {p.name}: {type(e).__name__}: {e}')
        raw_rows.append({
            'drawing': p.name,
            'source':  'failed',
            'total':   float('nan'),
        })

print(f'\n{len(raw_rows)} drawing(s) processed.')


[1/10] timing 104-11-2024ПВВ-ИГДИ-Г.3..dxf

=== 104-11-2024ПВВ-ИГДИ-Г.3..dxf ===
[ROUTE] Detected source: 'regions' (MSK matches: 0/2; total 48 layers in DXF)
[REG 1] Input: C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\time_test_topo\104-11-2024ПВВ-ИГДИ-Г.3..dxf
[REG 2] DXF: C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\time_test_topo\104-11-2024ПВВ-ИГДИ-Г.3..dxf
[REG 3] Extracted objects: 3058
[preproc] topo-codes loaded: 192 entries from C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\base_plan\2.0\topo_codes_classification.xlsx
[preproc] auto-named blocks normalised: 12
[preproc] dynamic blocks resolved:      0
[preproc] CAD-formatted texts cleaned:  13
[preproc] topo-code preassignments:     0
[REG 3.5] Preprocessing done (0 rows preassigned via topo codes)
Device: cpu
  regions: bert=intfloat/multilingual-e5-small, l1=11, l2=75 | mask 82/825 pairs
  loading tokenizer: intfloat/multilingual-e5-small
[REG 

In [ ]:
df_topo = pd.DataFrame(raw_rows)[['drawing', 'source', 'total']]
df_topo['total'] = df_topo['total'].round(2)
df_topo

,drawing,source,total
0,104-11-2024ПВВ-ИГДИ-Г.3..dxf,regions,171.17
1,106-11-2024ПВВ-ИГДИ-Г.3.dxf,regions,63.33
2,93-10-2024ПВВ-ИГДИ-Г.3.dxf,regions,112.65
3,ГЕО_Вишневый сад.dxf,regions,28.19
4,Зарайск_итог 10_02_2021_итог.dxf,regions,280.15
5,ИГДИ-Г.33213.dxf,regions,54.30
6,Лотошино готово.dxf,regions,146.01
7,Оленегорск ТГР.dxf,regions,5.94
8,Пушкино 500.dxf,regions,386.62
9,Топография.dxf,regions,227.61


In [11]:
# Cross-drawing summary stats (only meaningful with >1 file).
if len(raw_rows) > 1:
    df_summary = (df_topo['total']
                  .agg(['mean', 'median', 'min', 'max', 'std'])
                  .round(2)
                  .rename_axis('stat')
                  .reset_index(name='seconds'))
    display(df_summary)
else:
    df_summary = None
    print('Only one drawing — summary stats trivially equal the row; skipping.')

,stat,seconds
0,mean,147.60
1,median,129.33
2,min,5.94
3,max,386.62
4,std,121.57


In [12]:
# Save timings to a SEPARATE Excel file (parallel to design timings).
with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
    df_topo.to_excel(writer, sheet_name='timings', index=False)
    if df_summary is not None:
        df_summary.to_excel(writer, sheet_name='summary', index=False)

print(f'Saved: {EXCEL_PATH}')
print(f'  sheet "timings" — {len(df_topo)} rows')
if df_summary is not None:
    print(f'  sheet "summary" — {len(df_summary)} rows')

Saved: c:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\timings\topo_timings.xlsx
  sheet "timings" — 10 rows
  sheet "summary" — 5 rows


## Area statistics

Same idea as the area cell in `time_generation_design.ipynb`, but with
only one generated plan instead of four. Result lands in a SEPARATE
Excel file from the timings — `areas/topo_area_stats.xlsx`.

Time of the area-pass itself is **not** recorded — only the area values.

In [13]:
# Area stats: source DXF vs the generated topography DXF.
from calculate_area import calculate_dxf_area


def _safe_area(path):
    """calculate_dxf_area returns (path, value); coerce error / missing to NaN."""
    if not path.exists():
        return float('nan')
    _, value = calculate_dxf_area(str(path))
    if isinstance(value, (int, float)):
        return float(value)
    print(f'  ! {path.name}: {value}')
    return float('nan')


area_rows = []
for p in dxf_paths:
    out = p.parent / p.stem / f'{p.stem}_topo.dxf'
    print(f'Areas for {p.name}...')
    src_a  = _safe_area(p)
    topo_a = _safe_area(out)
    pct = (round(topo_a / src_a * 100.0, 1)
           if (src_a and src_a > 0 and not pd.isna(topo_a))
           else float('nan'))
    area_rows.append({
        'drawing':             p.name,
        'source_area':         round(src_a, 2) if not pd.isna(src_a) else src_a,
        'topo_area':           round(topo_a, 2) if not pd.isna(topo_a) else topo_a,
        'topo_pct_of_source':  pct,
    })

df_areas = pd.DataFrame(area_rows)

if len(area_rows) > 1:
    df_areas_summary = (df_areas[['source_area', 'topo_area']]
                        .agg(['mean', 'median', 'min', 'max', 'std'])
                        .round(2).T
                        .rename_axis('column').reset_index())
else:
    df_areas_summary = None

print('\n── Areas, m² (concave hull, calculate_dxf_area default settings) ──')
display(df_areas)
if df_areas_summary is not None:
    print('\n── Across-drawings summary ──')
    display(df_areas_summary)

with pd.ExcelWriter(AREAS_EXCEL_PATH, engine='openpyxl') as writer:
    df_areas.to_excel(writer, sheet_name='areas', index=False)
    if df_areas_summary is not None:
        df_areas_summary.to_excel(writer, sheet_name='summary', index=False)

print(f'\nSaved: {AREAS_EXCEL_PATH}')
print(f'  sheet "areas"   — {len(df_areas)} rows')
if df_areas_summary is not None:
    print(f'  sheet "summary" — {len(df_areas_summary)} rows')

Areas for 104-11-2024ПВВ-ИГДИ-Г.3..dxf...
Areas for 106-11-2024ПВВ-ИГДИ-Г.3.dxf...
Areas for 93-10-2024ПВВ-ИГДИ-Г.3.dxf...
Areas for ГЕО_Вишневый сад.dxf...
Areas for Зарайск_итог 10_02_2021_итог.dxf...
Areas for ИГДИ-Г.33213.dxf...
Areas for Лотошино готово.dxf...
Areas for Оленегорск ТГР.dxf...
Areas for Пушкино 500.dxf...
Areas for Топография.dxf...

── Areas, m² (concave hull, calculate_dxf_area default settings) ──


,drawing,source_area,topo_area,topo_pct_of_source
0,104-11-2024ПВВ-ИГДИ-Г.3..dxf,84505.20,77952.47,92.2
1,106-11-2024ПВВ-ИГДИ-Г.3.dxf,203229.43,171640.13,84.5
2,93-10-2024ПВВ-ИГДИ-Г.3.dxf,55191.67,496967.08,900.4
3,ГЕО_Вишневый сад.dxf,16719.72,50482.24,301.9
4,Зарайск_итог 10_02_2021_итог.dxf,1149985.33,1379032.33,119.9
5,ИГДИ-Г.33213.dxf,56053.39,102961.02,183.7
6,Лотошино готово.dxf,270881.39,252951.01,93.4
7,Оленегорск ТГР.dxf,4588.85,4441.87,96.8
8,Пушкино 500.dxf,998062.07,1070468.43,107.3
9,Топография.dxf,385795.28,436333.13,113.1



── Across-drawings summary ──


,column,mean,median,min,max,std
0,source_area,322501.23,143867.32,4588.85,1149985.33,415755.07
1,topo_area,404322.97,212295.57,4441.87,1379032.33,467013.60



Saved: c:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\design_plan\areas\topo_area_stats.xlsx
  sheet "areas"   — 10 rows
  sheet "summary" — 2 rows
